# Deep Learning PR5 — Complete Answer Notebook
## Sentiment Analysis with RNN, LSTM and GRU

**Student notebook template:** This notebook follows the uploaded PR5 assignment and submission checklist.

**Dataset required:** `IMDB Dataset.csv` downloaded from Kaggle. Keep the CSV in the same folder as this notebook. Do **not** use `keras.datasets.imdb`.

### Models covered
- Order-blind ANN baseline
- SimpleRNN
- LSTM
- Stacked LSTM
- GRU

### Main experiments
1. Text loading, cleaning, tokenisation and padding
2. RNN intuition, ANN baseline and manual NumPy forward pass
3. RNN types and SimpleRNN sentiment model
4. BPTT, vanishing/exploding gradients and gradient-flow experiments
5. LSTM theory, manual LSTM step, training and gradient comparison
6. GRU theory, training, gradient comparison and hidden-unit sensitivity
7. Final comparison, ROC curves, length analysis, error analysis, inference and recommendation

> Run the notebook top-to-bottom after placing `IMDB Dataset.csv` beside it. The assignment asks for `Restart & Run All` with zero errors.


## Task 0 — Imports, seeds and common configuration

The assignment requires reproducibility. We set NumPy, Python and TensorFlow seeds at the beginning.


In [ ]:
import os
import re
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("IMDB Dataset.csv")
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(exist_ok=True)

NUM_WORDS = 10_000
MAXLEN = 300
EMBED_DIM = 32
BATCH_SIZE = 64
EPOCHS = 6

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Dataset path:", DATA_PATH.resolve())


## Task 1 — Text Loading, Cleaning, Tokenisation & Padding

### 1.1 Load the IMDB CSV

The expected columns are `review` and `sentiment`. The assignment specifically requires the Kaggle CSV rather than `keras.datasets.imdb`.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "IMDB Dataset.csv was not found. Download the Kaggle IMDB Dataset.csv "
        "and place it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNull values:\n", df.isnull().sum())
print("\nClass balance:\n", df["sentiment"].value_counts())
display(df.head(2))


### 1.2 Duplicate removal and leakage explanation

Duplicates are counted before removal. Removing duplicates helps prevent the same review from appearing in both train and test sets, which can make test performance look artificially high.


In [ ]:
duplicate_count = df.duplicated(subset=["review"]).sum()
print("Duplicate reviews:", duplicate_count)

df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


### 1.3 Text cleaning

Requirements:
- remove HTML tags
- lowercase
- keep letters and spaces
- preserve apostrophes so contractions such as `don't` remain meaningful
- keep stop words because words such as `not` are important for sentiment


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_review"] = df["review"].apply(clean_text)

print("BEFORE:")
print(df.loc[0, "review"][:500])
print("\nAFTER:")
print(df.loc[0, "cleaned_review"][:500])


### Why keep stop words and apostrophes?
Stop words are kept because sentiment can depend on small words such as **not**, **never**, and **very**. Apostrophes are preserved because contractions such as **don't** and **isn't** contain sentiment-relevant negation information.


### 1.4 Review length analysis

The assignment requires a histogram with mean, median and 95th-percentile lines plus a sentiment boxplot.


In [ ]:
df["word_count"] = df["cleaned_review"].str.split().str.len()

mean_len = df["word_count"].mean()
median_len = df["word_count"].median()
p95_len = df["word_count"].quantile(0.95)

print(f"Mean length: {mean_len:.2f} words")
print(f"Median length: {median_len:.2f} words")
print(f"95th percentile: {p95_len:.2f} words")

plt.figure(figsize=(10, 5))
plt.hist(df["word_count"], bins=50)
plt.axvline(mean_len, linestyle="--", label=f"Mean = {mean_len:.1f}")
plt.axvline(median_len, linestyle="--", label=f"Median = {median_len:.1f}")
plt.axvline(p95_len, linestyle="--", label=f"95th percentile = {p95_len:.1f}")
plt.title("IMDB Review Length Distribution")
plt.xlabel("Review length (words)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "length_distribution.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="sentiment", y="word_count")
plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Word count")
plt.tight_layout()
plt.savefig(PLOT_DIR / "length_boxplot.png", dpi=150)
plt.show()


### 1.5 Train/test split BEFORE fitting the tokenizer

The assignment requires an 80/20 stratified split with `random_state=42`. The tokenizer must be fitted only on training text to avoid test-set vocabulary leakage.


In [ ]:
df["label"] = (df["sentiment"].str.lower() == "positive").astype(int)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["cleaned_review"],
    df["label"],
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

print("Train:", len(X_train_text))
print("Test :", len(X_test_text))
print("Train class balance:")
print(y_train.value_counts(normalize=True))
print("Test class balance:")
print(y_test.value_counts(normalize=True))


### 1.6 Tokenisation and padding

`num_words=10000` and an OOV token are required. Pre-padding is used because the assignment explicitly asks for `padding='pre'`. `MAXLEN=300` is a practical fixed sequence length for this project and is also used in the later sequence-length experiment.


In [ ]:
tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=NUM_WORDS,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train_text)

top10 = sorted(
    [(word, count) for word, count in tokenizer.word_counts.items()],
    key=lambda x: x[1],
    reverse=True
)[:10]
print("Top 10 training words:")
print(top10)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train = keras.preprocessing.sequence.pad_sequences(
    X_train_seq, maxlen=MAXLEN, padding="pre", truncating="pre"
)
X_test = keras.preprocessing.sequence.pad_sequences(
    X_test_seq, maxlen=MAXLEN, padding="pre", truncating="pre"
)

y_train = np.asarray(y_train)
y_test = np.asarray(y_test)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


### Why is text sequential?

Word order changes meaning. For example, **“not good”** is negative even though it contains the word “good”. An order-blind model that treats the text as an unordered collection can lose this relationship. RNNs process tokens in sequence and maintain a hidden state.


## Task 2 — RNN Intuition, ANN Baseline & Manual Forward Propagation


### 2.1 Order-blind ANN baseline

The baseline uses `Embedding → Flatten → Dense`. Flattening removes the explicit time-step structure, so this is an order-blind comparison against recurrent models.


In [ ]:
def build_ann():
    model = keras.Sequential([
        layers.Input(shape=(MAXLEN,)),
        layers.Embedding(NUM_WORDS, EMBED_DIM),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

ann = build_ann()
ann.summary()

start = time.perf_counter()
ann_hist = ann.fit(
    X_train, y_train,
    validation_split=0.10,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=1, restore_best_weights=True
    )],
    verbose=1
)
ann_time = (time.perf_counter() - start) / max(1, len(ann_hist.history["loss"]))

ann_prob = ann.predict(X_test, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)

print("ANN parameters:", ann.count_params())
print("ANN time/epoch:", round(ann_time, 4), "s")
print("ANN test accuracy:", accuracy_score(y_test, ann_pred))


### 2.2 RNN intuition

For a vanilla RNN, the hidden state is updated at each time step:

**h(t) = tanh(Wx(t) + Uh(t−1) + b)**

The same weights are reused at every time step. This weight sharing lets the network process variable-length sequences while carrying information forward through the hidden state.


### 2.3 Manual NumPy RNN forward pass — 4 steps, hidden size 3


In [ ]:
# Manual forward pass using the same convention as Keras SimpleRNN:
# h_t = tanh(x_t @ W_x + h_(t-1) @ W_h + b)

np.random.seed(SEED)
input_size = 2
hidden_size = 3
T = 4

x_manual = np.random.randn(T, input_size).astype(np.float32)
Wx = np.random.randn(input_size, hidden_size).astype(np.float32)
Wh = np.random.randn(hidden_size, hidden_size).astype(np.float32)
b = np.random.randn(hidden_size).astype(np.float32)

h = np.zeros(hidden_size, dtype=np.float32)
manual_states = []

for t in range(T):
    h = np.tanh(x_manual[t] @ Wx + h @ Wh + b)
    manual_states.append(h.copy())

manual_states = np.array(manual_states)

print(pd.DataFrame(
    manual_states,
    index=[f"t={i+1}" for i in range(T)],
    columns=[f"h{i+1}" for i in range(hidden_size)]
))

plt.figure(figsize=(7, 4))
sns.heatmap(manual_states, annot=True, fmt=".3f", cmap="viridis")
plt.title("Manual RNN Hidden-State Heatmap")
plt.xlabel("Hidden unit")
plt.ylabel("Time step")
plt.tight_layout()
plt.savefig(PLOT_DIR / "manual_forward_heatmap.png", dpi=150)
plt.show()


### 2.4 Verify manual forward pass against Keras SimpleRNN

The Keras layer is given the exact same weights and inputs. `np.allclose` checks that the two implementations agree within floating-point tolerance.


In [ ]:
keras_rnn = layers.SimpleRNN(
    hidden_size, activation="tanh", use_bias=True,
    return_sequences=True
)
_ = keras_rnn(tf.zeros((1, T, input_size)))

# Keras stores kernel as input -> hidden and recurrent_kernel as hidden -> hidden.
keras_rnn.set_weights([Wx, Wh, b])

keras_states = keras_rnn(
    tf.convert_to_tensor(x_manual[None, :, :])
).numpy()[0]

print("Manual states:")
print(manual_states)
print("\nKeras states:")
print(keras_states)
print("\nnp.allclose:", np.allclose(manual_states, keras_states, atol=1e-6))


### 2.5 SimpleRNN parameter-count formula

For `units = H` and input dimension `D`:

**Parameters = H × (D + H + 1)**

The `+1` is the bias for each hidden unit. For the project's `SimpleRNN(64)` with embedding dimension 32:

**64 × (32 + 64 + 1) = 6,208**

The exact model summary should be used to verify the count.


In [ ]:
simple_rnn_formula = 64 * (EMBED_DIM + 64 + 1)
print("Formula count for SimpleRNN(64):", simple_rnn_formula)

tmp = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(NUM_WORDS, EMBED_DIM),
    layers.SimpleRNN(64),
    layers.Dense(1, activation="sigmoid")
])
print("Keras SimpleRNN layer parameters:", tmp.layers[2].count_params())


### 2.6 RNN vs ANN comparison

| Property | ANN baseline | RNN |
|---|---|---|
| Input order | Mostly ignored after Flatten | Explicitly processed |
| Memory | No recurrent hidden state | Hidden state carries previous information |
| Weight sharing across time | No | Yes |
| Sequential dependency | Weak | Strong |
| Main limitation | Loses word order | Can suffer vanishing/exploding gradients |


### 2.7 Four RNN input/output patterns

- **One-to-one:** ordinary classification/regression.
- **One-to-many:** one input generates a sequence, e.g. image caption generation.
- **Many-to-one:** a sequence produces one output, e.g. sentiment analysis.
- **Many-to-many:** sequence-to-sequence tasks such as tagging or translation.

Sentiment analysis in this project is **many-to-one**.


In [ ]:
# Dummy-data demonstration of common RNN output shapes.
dummy = np.random.randn(2, 5, 8).astype(np.float32)

one_to_one = layers.Dense(3)(np.random.randn(2, 8).astype(np.float32))
one_to_many = layers.SimpleRNN(4, return_sequences=True)(
    np.random.randn(2, 5, 8).astype(np.float32)
)
many_to_one = layers.SimpleRNN(4, return_sequences=False)(dummy)
many_to_many = layers.SimpleRNN(4, return_sequences=True)(dummy)

print("One-to-one:", one_to_one.shape)
print("One-to-many:", one_to_many.shape)
print("Many-to-one:", many_to_one.shape)
print("Many-to-many:", many_to_many.shape)


## Task 3 — SimpleRNN Sentiment Analysis


### 3.1 Build, train and evaluate SimpleRNN


In [ ]:
def build_model(kind="simplernn", units=64):
    model = keras.Sequential([layers.Input(shape=(MAXLEN,))])
    model.add(layers.Embedding(NUM_WORDS, EMBED_DIM))

    if kind == "simplernn":
        model.add(layers.SimpleRNN(units))
    elif kind == "lstm":
        model.add(layers.LSTM(units))
    elif kind == "gru":
        model.add(layers.GRU(units))
    else:
        raise ValueError("kind must be simplernn, lstm or gru")

    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def evaluate_binary_model(model, X, y):
    prob = model.predict(X, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    metrics = {
        "Test Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y, prob)
    }
    return metrics, prob, pred

def train_model(model, name):
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_split=0.10,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )],
        verbose=1
    )
    elapsed = time.perf_counter() - start
    time_per_epoch = elapsed / max(1, len(history.history["loss"]))
    return history, time_per_epoch

simple_rnn = build_model("simplernn", 64)
simple_hist, simple_time = train_model(simple_rnn, "SimpleRNN")
simple_metrics, simple_prob, simple_pred = evaluate_binary_model(
    simple_rnn, X_test, y_test
)

print("SimpleRNN parameters:", simple_rnn.count_params())
print("Time/epoch:", simple_time)
print(simple_metrics)
print("\nClassification report:")
print(classification_report(y_test, simple_pred, target_names=["negative", "positive"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, simple_pred))


In [ ]:
def plot_history(history, title, filename):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.history["accuracy"], label="Train accuracy")
    ax.plot(history.history["val_accuracy"], label="Validation accuracy")
    ax.set_title(title + " — Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / filename, dpi=150)
    plt.show()

plot_history(simple_hist, "SimpleRNN", "training_curves_simplernn.png")

cm = confusion_matrix(y_test, simple_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"],
            yticklabels=["negative", "positive"])
plt.title("SimpleRNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cm_simplernn.png", dpi=150)
plt.show()


### 3.2 Embedding layer explanation

The embedding layer converts integer token IDs into dense vectors. In this project, each token is represented by a 32-dimensional learned vector. Similar words can learn similar vector representations because the vectors are updated during sentiment training.


In [ ]:
embedding_weights = simple_rnn.layers[1].get_weights()[0]
print("Embedding matrix shape:", embedding_weights.shape)

word_index = tokenizer.word_index
for word in ["good", "bad"]:
    idx = word_index.get(word)
    if idx is not None and idx < NUM_WORDS:
        print(word, "-> first 10 embedding values:",
              np.round(embedding_weights[idx][:10], 4))


## Task 4 — Backpropagation Through Time & RNN Problems


### 4.1 BPTT and the product-of-Jacobians

During Backpropagation Through Time, the gradient at an early time step depends on repeated Jacobian products:

**∂L/∂hₜ = (∂L/∂hₜ₊₁)(∂hₜ₊₁/∂hₜ)**

Across many time steps this becomes a product of many matrices. If their effective magnitudes are repeatedly below 1, gradients can vanish. If they are repeatedly above 1, gradients can explode.


### 4.2 Vanishing vs exploding gradients

**Vanishing gradient:** gradient norms become very small at earlier time steps, making long-range learning difficult.

**Exploding gradient:** gradient norms become very large, causing unstable parameter updates or NaNs.

Gradient clipping limits excessively large gradients. It does not solve the underlying long-term memory problem of a plain RNN.


### 4.3 Per-time-step gradient norms with GradientTape

The recurrent model below returns every hidden state. We use a persistent tape to measure how strongly the final loss depends on each time step.


In [ ]:
# A smaller model makes the gradient experiment faster and easier to inspect.
grad_rnn = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(NUM_WORDS, EMBED_DIM),
    layers.SimpleRNN(64, return_sequences=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(1, activation="sigmoid")
])

loss_fn = keras.losses.BinaryCrossentropy()

sample_n = min(128, len(X_test))
x_grad = tf.convert_to_tensor(X_test[:sample_n])
y_grad = tf.convert_to_tensor(y_test[:sample_n, None], dtype=tf.float32)

with tf.GradientTape() as tape:
    tape.watch(x_grad)
    outputs = grad_rnn(x_grad, training=False)

# Gradient with respect to the integer token IDs is not useful for an embedding.
# Instead measure gradient through the embedding output by building a functional model.
inp = keras.Input(shape=(MAXLEN,), dtype="int32")
emb_out = layers.Embedding(NUM_WORDS, EMBED_DIM)(inp)
rnn_seq = layers.SimpleRNN(64, return_sequences=True)(emb_out)
pool = layers.GlobalAveragePooling1D()(rnn_seq)
out = layers.Dense(1, activation="sigmoid")(pool)
grad_model = keras.Model(inp, [emb_out, rnn_seq, out])

with tf.GradientTape() as tape:
    emb_values, rnn_values, pred_values = grad_model(x_grad, training=False)
    loss_value = loss_fn(y_grad, pred_values)

grads = tape.gradient(loss_value, rnn_values)
grad_norm_by_time = tf.reduce_mean(
    tf.norm(grads, axis=-1), axis=0
).numpy()

plt.figure(figsize=(9, 4))
plt.plot(range(1, MAXLEN + 1), grad_norm_by_time)
plt.yscale("log")
plt.title("SimpleRNN Gradient Norm by Time Step")
plt.xlabel("Time step")
plt.ylabel("Mean gradient norm (log scale)")
plt.tight_layout()
plt.savefig(PLOT_DIR / "gradient_norms_simplernn.png", dpi=150)
plt.show()


### 4.4 Accuracy vs sequence length

The experiment uses sequence lengths 50, 100, 200 and 300 as required. We train a small SimpleRNN for a short number of epochs to compare the effect of retained sequence length.


In [ ]:
def make_padded_subset(sequences, maxlen):
    return keras.preprocessing.sequence.pad_sequences(
        sequences, maxlen=maxlen, padding="pre", truncating="pre"
    )

sequence_lengths = [50, 100, 200, 300]
length_results = []

for L in sequence_lengths:
    Xtr_L = make_padded_subset(X_train_seq, L)
    Xte_L = make_padded_subset(X_test_seq, L)

    m = keras.Sequential([
        layers.Input(shape=(L,)),
        layers.Embedding(NUM_WORDS, EMBED_DIM),
        layers.SimpleRNN(32),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(
        Xtr_L, y_train,
        validation_split=0.10,
        epochs=2,
        batch_size=BATCH_SIZE,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )],
        verbose=0
    )
    loss, acc = m.evaluate(Xte_L, y_test, verbose=0)
    length_results.append({"MAXLEN": L, "Test Accuracy": acc})

length_df = pd.DataFrame(length_results)
display(length_df)

plt.figure(figsize=(7, 4))
plt.plot(length_df["MAXLEN"], length_df["Test Accuracy"], marker="o")
plt.title("SimpleRNN Accuracy vs Sequence Length")
plt.xlabel("Maximum sequence length")
plt.ylabel("Test accuracy")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(PLOT_DIR / "accuracy_vs_sequence_length.png", dpi=150)
plt.show()


### 4.5 Exploding-gradient demonstration with and without clipping

A deliberately high learning rate is used to make instability more visible. The second run uses `clipnorm=1.0`. Clipping is a numerical stabilisation method for exploding gradients.


In [ ]:
def exploding_gradient_demo(clipnorm=None):
    model = keras.Sequential([
        layers.Input(shape=(MAXLEN,)),
        layers.Embedding(NUM_WORDS, EMBED_DIM),
        layers.SimpleRNN(32),
        layers.Dense(1, activation="sigmoid")
    ])
    opt = keras.optimizers.Adam(
        learning_rate=0.05,
        clipnorm=clipnorm
    )
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    hist = model.fit(
        X_train[:8000], y_train[:8000],
        validation_data=(X_test[:2000], y_test[:2000]),
        epochs=3,
        batch_size=128,
        verbose=0
    )
    return hist

hist_no_clip = exploding_gradient_demo(None)
hist_clip = exploding_gradient_demo(1.0)

plt.figure(figsize=(8, 4))
plt.plot(hist_no_clip.history["loss"], label="No clipping")
plt.plot(hist_clip.history["loss"], label="clipnorm=1.0")
plt.title("Exploding-Gradient Demo: Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "exploding_gradient_clipping.png", dpi=150)
plt.show()


### 4.6 Problems summary

| Problem | Cause | Symptom | Remedy |
|---|---|---|---|
| Vanishing gradient | Repeated small Jacobian factors | Early time steps learn weakly | LSTM/GRU gating |
| Exploding gradient | Repeated large factors | Unstable loss/very large updates | Gradient clipping |
| Slow sequential training | Recurrent dependence across time | Training is less parallel | Smaller sequences/GPU |
| Short-term memory | Plain hidden state struggles with long dependencies | Long reviews may degrade | Gated cells such as LSTM/GRU |


## Task 5 — LSTM: Long Short-Term Memory


### 5.1 LSTM theory

Using the notation requested in the assignment:

- **Forget gate:** `f(t) = sigmoid(Wf [h(t−1), x(t)] + bf)`
- **Input gate:** `i(t) = sigmoid(Wi [h(t−1), x(t)] + bi)`
- **Candidate state:** `g(t) = tanh(Wg [h(t−1), x(t)] + bg)`
- **Cell update:** `C(t) = f(t) ⊙ C(t−1) + i(t) ⊙ g(t)`
- **Output gate:** `o(t) = sigmoid(Wo [h(t−1), x(t)] + bo)`
- **Hidden state:** `h(t) = o(t) ⊙ tanh(C(t))`

Plain-language interpretation:
- forget gate decides what old information to discard
- input gate decides what new information to write
- candidate state proposes new content
- cell state carries long-term information
- output gate decides what part of the cell state becomes the visible hidden state


### 5.2 Why LSTM mitigates vanishing gradients

The cell-state update contains an additive path. When the forget gate is close to 1, information can pass across time with much less attenuation than in a plain RNN where the hidden-state gradient repeatedly passes through recurrent weights and nonlinearities.


### 5.3 LSTM parameter count

For input dimension `D` and `H` units:

**Parameters = 4 × H × (D + H + 1)**

For `H=64`, `D=32`:

**4 × 64 × (32 + 64 + 1) = 24,832**

This is four times the SimpleRNN recurrent-layer count because an LSTM has four learned affine transformations/gate sets.


In [ ]:
lstm_formula = 4 * 64 * (EMBED_DIM + 64 + 1)
print("LSTM(64) formula count:", lstm_formula)

tmp_lstm = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(NUM_WORDS, EMBED_DIM),
    layers.LSTM(64),
    layers.Dense(1, activation="sigmoid")
])
print("Keras LSTM layer parameters:", tmp_lstm.layers[2].count_params())


### 5.4 Manual NumPy LSTM step — 4 time steps

The equations below use the assignment's `f, i, g, o` notation. Random weights use a fixed seed.


In [ ]:
np.random.seed(SEED)

D, H, T = 2, 3, 4
x_seq = np.random.randn(T, D).astype(np.float32)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# Each gate receives [x_t, h_(t-1)]
Wf = np.random.randn(D + H, H).astype(np.float32)
Wi = np.random.randn(D + H, H).astype(np.float32)
Wg = np.random.randn(D + H, H).astype(np.float32)
Wo = np.random.randn(D + H, H).astype(np.float32)

bf = np.random.randn(H).astype(np.float32)
bi = np.random.randn(H).astype(np.float32)
bg = np.random.randn(H).astype(np.float32)
bo = np.random.randn(H).astype(np.float32)

h = np.zeros(H, dtype=np.float32)
C = np.zeros(H, dtype=np.float32)
rows = []

for t in range(T):
    z = np.concatenate([x_seq[t], h])
    f = sigmoid(z @ Wf + bf)
    i = sigmoid(z @ Wi + bi)
    g = np.tanh(z @ Wg + bg)
    o = sigmoid(z @ Wo + bo)
    C = f * C + i * g
    h = o * np.tanh(C)
    rows.append(np.concatenate([f, i, g, o, C, h]))

cols = (
    [f"f{i+1}" for i in range(H)] +
    [f"i{i+1}" for i in range(H)] +
    [f"g{i+1}" for i in range(H)] +
    [f"o{i+1}" for i in range(H)] +
    [f"C{i+1}" for i in range(H)] +
    [f"h{i+1}" for i in range(H)]
)
display(pd.DataFrame(rows, index=[f"t={i+1}" for i in range(T)], columns=cols))


### 5.5 Train and evaluate LSTM


In [ ]:
lstm = build_model("lstm", 64)
lstm_hist, lstm_time = train_model(lstm, "LSTM")
lstm_metrics, lstm_prob, lstm_pred = evaluate_binary_model(
    lstm, X_test, y_test
)

print("LSTM parameters:", lstm.count_params())
print("Time/epoch:", lstm_time)
print(lstm_metrics)
print("\nClassification report:")
print(classification_report(y_test, lstm_pred, target_names=["negative", "positive"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, lstm_pred))

plot_history(lstm_hist, "LSTM", "training_curves_lstm.png")


### 5.6 Gradient-flow comparison: SimpleRNN vs LSTM

The function below builds a model whose recurrent layer returns the full sequence, then computes gradient norms through the recurrent output. Both curves are plotted on the same log-scaled axes.


In [ ]:
def recurrent_gradient_curve(cell_layer, max_samples=128):
    inp = keras.Input(shape=(MAXLEN,), dtype="int32")
    emb = layers.Embedding(NUM_WORDS, EMBED_DIM)(inp)
    seq = cell_layer(emb)
    out = layers.Dense(1, activation="sigmoid")(seq[:, -1, :])
    model = keras.Model(inp, [seq, out])

    xs = tf.convert_to_tensor(X_test[:max_samples])
    ys = tf.convert_to_tensor(y_test[:max_samples, None], dtype=tf.float32)

    with tf.GradientTape() as tape:
        seq_values, pred = model(xs, training=False)
        loss = loss_fn(ys, pred)
    grads = tape.gradient(loss, seq_values)
    return tf.reduce_mean(tf.norm(grads, axis=-1), axis=0).numpy()

rnn_grad = recurrent_gradient_curve(
    layers.SimpleRNN(64, return_sequences=True)
)
lstm_grad = recurrent_gradient_curve(
    layers.LSTM(64, return_sequences=True)
)

plt.figure(figsize=(9, 4))
plt.plot(rnn_grad, label="SimpleRNN")
plt.plot(lstm_grad, label="LSTM")
plt.yscale("log")
plt.title("Gradient Norm: SimpleRNN vs LSTM")
plt.xlabel("Time step")
plt.ylabel("Gradient norm (log scale)")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "gradient_norms_rnn_lstm.png", dpi=150)
plt.show()

early_n = min(50, len(rnn_grad))
rnn_early = float(np.mean(rnn_grad[:early_n]))
lstm_early = float(np.mean(lstm_grad[:early_n]))
print("Mean earliest-step gradient — SimpleRNN:", rnn_early)
print("Mean earliest-step gradient — LSTM:", lstm_early)
if rnn_early > 0:
    print("LSTM / SimpleRNN earliest-gradient ratio:", lstm_early / rnn_early)


### 5.7 Stacked LSTM

The first LSTM must use `return_sequences=True` so that it outputs a complete sequence for the second LSTM.


In [ ]:
stacked_lstm = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(NUM_WORDS, EMBED_DIM),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])
stacked_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

stacked_hist, stacked_time = train_model(stacked_lstm, "Stacked LSTM")
stacked_metrics, stacked_prob, stacked_pred = evaluate_binary_model(
    stacked_lstm, X_test, y_test
)

print("Stacked LSTM parameters:", stacked_lstm.count_params())
print("Time/epoch:", stacked_time)
print(stacked_metrics)


## Task 6 — GRU: Gated Recurrent Unit


### 6.1 GRU theory

Using the assignment's notation:

- **Update gate:** `z(t) = sigmoid(Wz [h(t−1), x(t)] + bz)`
- **Reset gate:** `r(t) = sigmoid(Wr [h(t−1), x(t)] + br)`
- **Candidate state:** `h~(t) = tanh(Wh [r(t) ⊙ h(t−1), x(t)] + bh)`
- **New state:** `h(t) = (1 − z(t)) ⊙ h(t−1) + z(t) ⊙ h~(t)`

The update gate decides how much old information to keep versus replace. The reset gate decides how much previous information contributes when creating the candidate state.


### 6.2 GRU vs LSTM

| Feature | GRU | LSTM |
|---|---|---|
| Gates | 2 main gates | 3 gates |
| Separate cell state | No | Yes |
| Main state | Hidden state | Cell state + hidden state |
| Parameters | Usually fewer | Usually more |
| Training speed | Often faster | Often slower |
| Long dependencies | Strong | Strong |
| Typical choice | Efficient model with fewer parameters | When explicit memory control is useful |

A GRU merges the forget/input decision into the update gate.


### 6.3 GRU parameter count

For Keras `GRU(..., reset_after=True)`, the assignment specifies:

**Parameters = 3 × (H × D + H × H + 2H)**

For `H=64`, `D=32`:

**3 × (2,048 + 4,096 + 128) = 18,816**


In [ ]:
gru_formula = 3 * (64 * EMBED_DIM + 64 * 64 + 2 * 64)
print("GRU(64) formula count:", gru_formula)

tmp_gru = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(NUM_WORDS, EMBED_DIM),
    layers.GRU(64, reset_after=True),
    layers.Dense(1, activation="sigmoid")
])
print("Keras GRU layer parameters:", tmp_gru.layers[2].count_params())


### 6.4 Train and evaluate GRU


In [ ]:
gru = build_model("gru", 64)
gru_hist, gru_time = train_model(gru, "GRU")
gru_metrics, gru_prob, gru_pred = evaluate_binary_model(
    gru, X_test, y_test
)

print("GRU parameters:", gru.count_params())
print("Time/epoch:", gru_time)
print(gru_metrics)
print("\nClassification report:")
print(classification_report(y_test, gru_pred, target_names=["negative", "positive"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, gru_pred))

plot_history(gru_hist, "GRU", "training_curves_gru.png")


### 6.5 Gradient flow — SimpleRNN, LSTM and GRU


In [ ]:
gru_grad = recurrent_gradient_curve(
    layers.GRU(64, return_sequences=True)
)

plt.figure(figsize=(9, 4))
plt.plot(rnn_grad, label="SimpleRNN")
plt.plot(lstm_grad, label="LSTM")
plt.plot(gru_grad, label="GRU")
plt.yscale("log")
plt.title("Gradient Norm Comparison: SimpleRNN vs LSTM vs GRU")
plt.xlabel("Time step")
plt.ylabel("Gradient norm (log scale)")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "gradient_norms_snn_lstm_gru.png", dpi=150)
plt.show()


### 6.6 Hidden-unit sensitivity: 32, 64 and 128

The assignment asks for LSTM and GRU accuracy and parameter-count comparisons across 32, 64 and 128 hidden units. To keep the experiment practical, this cell uses the same training pipeline but can be reduced to fewer epochs if your computer is slow.


In [ ]:
sensitivity_rows = []

for kind in ["lstm", "gru"]:
    for units in [32, 64, 128]:
        m = build_model(kind, units)
        start = time.perf_counter()
        h = m.fit(
            X_train, y_train,
            validation_split=0.10,
            epochs=2,
            batch_size=BATCH_SIZE,
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=1, restore_best_weights=True
            )],
            verbose=0
        )
        elapsed = time.perf_counter() - start
        metrics, _, _ = evaluate_binary_model(m, X_test, y_test)
        sensitivity_rows.append({
            "Model": kind.upper(),
            "Units": units,
            "Parameters": m.count_params(),
            "Test Accuracy": metrics["Test Accuracy"],
            "Time/Epoch (s)": elapsed / max(1, len(h.history["loss"]))
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df)

plt.figure(figsize=(8, 4))
for kind in ["LSTM", "GRU"]:
    sub = sensitivity_df[sensitivity_df["Model"] == kind]
    plt.plot(sub["Units"], sub["Test Accuracy"], marker="o", label=kind)
plt.title("Test Accuracy vs Hidden Units")
plt.xlabel("Hidden units")
plt.ylabel("Test accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "accuracy_by_hidden_units.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
for kind in ["LSTM", "GRU"]:
    sub = sensitivity_df[sensitivity_df["Model"] == kind]
    plt.plot(sub["Units"], sub["Parameters"], marker="o", label=kind)
plt.title("Parameter Count vs Hidden Units")
plt.xlabel("Hidden units")
plt.ylabel("Parameters")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "parameters_by_hidden_units.png", dpi=150)
plt.show()


## Task 7 — Final Comparison, Inference & Recommendation


### 7.1 Full results table

The five required models are ANN baseline, SimpleRNN, LSTM, Stacked LSTM and GRU.


In [ ]:
results = {
    "ANN baseline": {
        "Parameters": ann.count_params(),
        "Time per Epoch (s)": ann_time,
        "Epochs Trained": len(ann_hist.history["loss"]),
        **ann_metrics
    },
    "SimpleRNN": {
        "Parameters": simple_rnn.count_params(),
        "Time per Epoch (s)": simple_time,
        "Epochs Trained": len(simple_hist.history["loss"]),
        **simple_metrics
    },
    "LSTM": {
        "Parameters": lstm.count_params(),
        "Time per Epoch (s)": lstm_time,
        "Epochs Trained": len(lstm_hist.history["loss"]),
        **lstm_metrics
    },
    "Stacked LSTM": {
        "Parameters": stacked_lstm.count_params(),
        "Time per Epoch (s)": stacked_time,
        "Epochs Trained": len(stacked_hist.history["loss"]),
        **stacked_metrics
    },
    "GRU": {
        "Parameters": gru.count_params(),
        "Time per Epoch (s)": gru_time,
        "Epochs Trained": len(gru_hist.history["loss"]),
        **gru_metrics
    }
}

results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
display(results_df.style.highlight_max(
    subset=["Test Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    color="#d9f2d9"
))

results_df.to_csv(PLOT_DIR / "results_table.csv", index=False)


### 7.2 Combined ROC curves

ROC curves show the trade-off between true-positive rate and false-positive rate over many classification thresholds. Accuracy alone reports performance at one threshold and can hide threshold-dependent behaviour.


In [ ]:
roc_models = {
    "ANN": ann_prob,
    "SimpleRNN": simple_prob,
    "LSTM": lstm_prob,
    "GRU": gru_prob
}

plt.figure(figsize=(8, 6))
for name, prob in roc_models.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("ROC Curves — ANN, SimpleRNN, LSTM and GRU")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "roc_curves.png", dpi=150)
plt.show()


### 7.3 Accuracy by review length

Definitions required by the assignment:
- Short: below 100 words
- Medium: 100–250 words
- Long: above 250 words

The best gated model is selected by test F1. It is compared with SimpleRNN.


In [ ]:
best_gated_name = max(
    ["LSTM", "GRU"],
    key=lambda n: results_df.loc[results_df["Model"] == n, "F1"].iloc[0]
)
best_gated_model = lstm if best_gated_name == "LSTM" else gru
best_gated_prob = lstm_prob if best_gated_name == "LSTM" else gru_prob
best_gated_pred = lstm_pred if best_gated_name == "LSTM" else gru_pred

test_length = X_test_text.str.split().str.len().to_numpy()

def bucket_length(n):
    if n < 100:
        return "Short"
    if n <= 250:
        return "Medium"
    return "Long"

length_groups = np.array([bucket_length(n) for n in test_length])

rows = []
for model_name, probs, preds in [
    ("SimpleRNN", simple_prob, simple_pred),
    (best_gated_name, best_gated_prob, best_gated_pred)
]:
    for group in ["Short", "Medium", "Long"]:
        mask = length_groups == group
        rows.append({
            "Model": model_name,
            "Length": group,
            "Accuracy": accuracy_score(y_test[mask], preds[mask]) if mask.any() else np.nan
        })

length_model_df = pd.DataFrame(rows)
display(length_model_df)

pivot = length_model_df.pivot(index="Length", columns="Model", values="Accuracy")
pivot.plot(kind="bar", figsize=(8, 4))
plt.title("Accuracy by Review Length")
plt.xlabel("Review length")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(PLOT_DIR / "accuracy_by_length.png", dpi=150)
plt.show()


### 7.4 Error analysis — 4 false positives and 4 false negatives

A false positive is a negative review predicted as positive. A false negative is a positive review predicted as negative. The code prints the true label, predicted probability and cleaned review.


In [ ]:
error_df = pd.DataFrame({
    "review": X_test_text.values,
    "true": y_test,
    "probability": best_gated_prob,
    "pred": best_gated_pred
})

fp = error_df[(error_df["true"] == 0) & (error_df["pred"] == 1)].head(4)
fn = error_df[(error_df["true"] == 1) & (error_df["pred"] == 0)].head(4)

print("FALSE POSITIVES")
display(fp)

print("FALSE NEGATIVES")
display(fn)


### Error-analysis discussion

When you run the cell above, inspect the eight examples for:
- sarcasm
- negation
- mixed positive/negative opinions
- very long reviews
- descriptions of a bad character or event inside an otherwise positive film
- subtle contextual cues

The discussion should describe patterns actually observed in your eight printed examples rather than inventing patterns.


### 7.5 Inference function

The inference function must reuse the exact training cleaning, tokenizer and padding pipeline.


In [ ]:
def predict_sentiment(text, model=best_gated_model):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = keras.preprocessing.sequence.pad_sequences(
        seq, maxlen=MAXLEN, padding="pre", truncating="pre"
    )
    probability = float(model.predict(padded, verbose=0)[0][0])
    label = "positive" if probability >= 0.5 else "negative"
    return label, probability

custom_sentences = [
    "This movie was absolutely wonderful and I loved every minute of it.",
    "This was a terrible and boring movie.",
    "This film is not good.",
    "I don't dislike this movie, it is actually enjoyable.",
    "Oh great, another masterpiece of two hours of boredom.",
    "Good."
]

custom_rows = []
for text in custom_sentences:
    label, probability = predict_sentiment(text)
    custom_rows.append({
        "Text": text,
        "Predicted Label": label,
        "Probability Positive": probability
    })

custom_df = pd.DataFrame(custom_rows)
display(custom_df)


### 7.6 Which recurrent cell should we use?

Use the **actual results table and experiments from this notebook** to complete the final recommendation.

A strong evidence-based structure is:

1. **Accuracy, F1 and ROC-AUC:** identify the model with the best observed scores.
2. **Training time and parameters:** compare accuracy gains with computational cost.
3. **Long sequences:** use the short/medium/long chart and gradient-flow experiment.
4. **Production recommendation:** select the model that gives the best practical balance of predictive quality, stability and cost.
5. **Limitations:** mention observations only, for example lack of pretrained knowledge and one-directional reading.

Do not claim a model is best before looking at your executed results.


In [ ]:
# Automatically generate a compact recommendation from the observed results.
metric_cols = ["Test Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
best_by_f1 = results_df.loc[results_df["F1"].idxmax(), "Model"]
best_by_auc = results_df.loc[results_df["ROC-AUC"].idxmax(), "Model"]

print("Best model by F1:", best_by_f1)
print("Best model by ROC-AUC:", best_by_auc)
print("\nRecommendation template:")
print(
    f"Based on the executed results, {best_by_f1} has the highest observed F1, "
    f"while {best_by_auc} has the highest observed ROC-AUC. "
    "The final production choice should also consider parameters, training time, "
    "and performance on long reviews."
)


### 7.7 Notebook-quality checks

Before submission:
- Set seeds.
- Every task starts with an opening Markdown cell.
- All plots have titles, axis labels and legends where applicable.
- Run **Kernel → Restart & Run All** with zero errors.
- Export the notebook to HTML.
- Keep `requirements.txt` committed.


In [ ]:
print("Random seed:", SEED)
print("Plots directory:", PLOT_DIR.resolve())
print("Required files to submit:")
for p in [
    "DL_PR5_Complete_Answer.ipynb",
    "requirements.txt",
    "README.md",
    "plots/"
]:
    print(" -", p)


# Submission checklist

- [ ] `IMDB Dataset.csv` downloaded from Kaggle
- [ ] Shape, class balance, null check and raw reviews shown
- [ ] Duplicates counted and removed
- [ ] `clean_text()` implemented and before/after example shown
- [ ] Stop-word/apostrophe explanation included
- [ ] Length histogram and sentiment boxplot created
- [ ] 80/20 stratified split performed before tokenizer fitting
- [ ] Tokenizer fitted on training text only
- [ ] `MAXLEN=300`, pre-padding and final shapes shown
- [ ] ANN baseline trained
- [ ] RNN equation and weight sharing explained
- [ ] Manual NumPy RNN forward pass and heatmap shown
- [ ] Manual RNN verified against Keras with `np.allclose`
- [ ] SimpleRNN parameter formula verified
- [ ] Four RNN types and shapes demonstrated
- [ ] SimpleRNN trained/evaluated with accuracy, precision, recall, F1 and ROC-AUC
- [ ] BPTT, vanishing and exploding gradients explained
- [ ] GradientTape gradient-norm plot completed
- [ ] Sequence-length experiment completed
- [ ] Exploding-gradient clipping experiment completed
- [ ] LSTM equations and parameter formula verified
- [ ] Manual NumPy LSTM step completed
- [ ] LSTM trained/evaluated
- [ ] Stacked LSTM trained/evaluated
- [ ] GRU equations and parameter formula verified
- [ ] GRU trained/evaluated
- [ ] Three-way gradient comparison completed
- [ ] Hidden-unit sensitivity completed
- [ ] Five-model results table created
- [ ] Combined ROC curves created
- [ ] Accuracy-by-length comparison created
- [ ] Eight misclassified reviews analysed
- [ ] `predict_sentiment()` tested on 6+ custom sentences
- [ ] Recommendation section completed from actual results
- [ ] Notebook restarted and run with zero errors
- [ ] HTML exported
- [ ] Video recorded: face + screen, 5–10 minutes
- [ ] Public GitHub repository contains notebook, HTML, plots, README and requirements
- [ ] At least 3 meaningful Git commits
